# imports

In [ ]:

import sionna.rt as rt

# Other imports
import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np
import tensorflow as tf
from metrics import compute_kpis
from sim import simulate, build_rx_path

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies

# Query the Mitsuba variant that was automatically selected by Sionna RT.
# There should be no need to adjust it, but if you do, make sure to select
# a variant ending in `*_ad_mono_polarized`.
print(f"Mitsuba variant: {mi.variant()}")
print(tf.config.list_physical_devices('GPU'))

# Params

In [ ]:
preview = False;
render = True;
generate_metrics = True;

#single tone or multi tone signal
#filtering and bpsk symbols

# create scene

In [ ]:
scene = load_scene(rt.scene.simple_street_canyon, merge_shapes=False)
scene.frequency = 28e9

# create transmitter and add it to the scene
scene.tx_array = PlanarArray(num_rows=8, num_cols=8, pattern="tr38901", polarization="V") #document these
scene.rx_array = scene.tx_array



In [ ]:

# -----------------------------
# DEFINE CAMERAS
# -----------------------------
cameras = {
    #"cam1": Camera(position=[120, 5, 50], look_at=[15, 0, 0]),
    #,
    "cam2": Camera(position=[0, 0, 270], look_at=[0, 0, 0])
}

#------------------------------
# DEFINE DRONES
# -----------------------------
uas1 = rt.Receiver(name="uas_1", position=[-75.0, -5.0, 20.0], orientation=[0.0, 0.0, 0.0])
uas1.color=[1,1,0]
scene.add(uas1)
uas2 = rt.Receiver(name="uas_2", position=[-75.0, 0.0, 20.0], orientation=[0.0, 0.0, 0.0])
uas2.color=[1,0,1]
scene.add(uas2)
uas3 = rt.Receiver(name="uas_3", position=[-75.0, 5.0, 20.0], orientation=[0.0, 0.0, 0.0])
uas3.color=[0,1,1]
scene.add(uas3)

# create transmitter and add it to the scene
tx = rt.Transmitter(name="tx_1", position=[0.0, 0.0, 1.5])
scene.add(tx)

# -----------------------------
# CONFIG
# -----------------------------
num_drones = 3
num_steps = 5  # increased resolution
dt = 1        # smaller timestep

# -----------------------------
# BUILD RX PATHS [N, T, 3]
# -----------------------------
start_points = tf.constant([
    [-75, -5, 20],  # yellow
    [-75,  0, 20],  # cyan
    [-75,  5, 20]   # pink
], dtype=tf.float32)

base_velocities = tf.constant([
    [[0, -50,  0], [50, 0, 0], [50, 0, 0], [50, 0, 0], [00, 50, 0]],
    [[50, 0,  0], [0, 50, 0], [50, 0, 20], [0, -50, 0], [50, 0, 0]],
    [[25, 25, 15], [50, -50, 20], [0, 50, 0], [50, -50, -20], [0, 50, 0]]
], dtype=tf.float32)  # Shape: [N, _base_steps, 3]

rx_path = build_rx_path(start_points, base_velocities, num_steps, dt, on_mismatch="error")

In [ ]:
"""# 30 drones of randomness
# -----------------------------
# CONFIG
# -----------------------------
num_drones = 30
num_steps = 5
dt = 1.0



# -----------------------------
# BUILD TRANSMITTER PATHS [N, T, 3]
# -----------------------------
start_points = tf.constant([
    [-80, -50, 20],
    [-75, -40, 25],
    [-70, -30, 15],
    [-65, -20, 30],
    [-60, -10, 10],
    [-55,   0, 20],
    [-50,  10, 25],
    [-45,  20, 15],
    [-40,  30, 35],
    [-35,  40, 20],
    [-30,  50, 10],
    [-20, -50, 25],
    [-10, -40, 15],
    [  0, -30, 30],
    [ 10, -20, 20],
    [ 20, -10, 10],
    [ 30,   0, 25],
    [ 40,  10, 15],
    [ 50,  20, 35],
    [ 60,  30, 20],
    [ 70,  40, 10],
    [ 80,  50, 25],
    [-85,  55, 15],
    [-60,  55, 30],
    [-35, -55, 20],
    [ -5,  55, 10],
    [ 25, -55, 25],
    [ 55,  55, 15],
    [ 85, -55, 35],
    [  0,   0, 20]
], dtype=tf.float32)

# -----------------------------
# DEFINE DRONES
# -----------------------------
drone_colors = [
    [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1],
    [0,1,1], [1,0.5,0], [0.5,0,1], [0,0.5,1], [0.5,1,0],
    [1,0,0.5], [0,1,0.5], [0.7,0.7,0.7], [1,0.7,0], [0.7,1,0],
    [0,0.7,1], [1,0,0.7], [0.7,0,1], [0,1,0.7], [0.5,0.5,1],
    [1,0.5,0.5], [0.5,1,0.5], [0.5,0.5,0.5], [1,1,0.5], [1,0.5,1],
    [0.5,1,1], [0.8,0.2,0.2], [0.2,0.8,0.2], [0.2,0.2,0.8], [1,1,1]
]

uas_list = []

for i in range(30):
    uas = rt.Transmitter(
        name=f"uas_{i+1}",
        position=start_points[i].numpy().tolist(),
        orientation=[0.0, 0.0, 0.0]
    )
    uas.color = drone_colors[i]
    scene.add(uas)
    uas_list.append(uas)

# create receiver and add it to the scene
rx = rt.Receiver(name="rx_1", position=[0.0, 0.0, 1.5])
scene.add(rx)

velocities = tf.constant([
    [[ 4,  2,  1], [ 5, -3,  0], [ 3,  4, -1], [ 6,  0,  2], [ 4, -2,  1]],
    [[ 5,  1, -1], [ 4,  3,  2], [ 3, -4,  0], [ 5,  2,  1], [ 4, -3, -1]],
    [[ 3,  5,  1], [ 4, -2,  2], [ 5,  1, -1], [ 3,  4,  0], [ 4, -3,  1]],
    [[ 6, -1,  0], [ 4,  3,  1], [ 5, -2,  2], [ 3,  5, -1], [ 4, -2,  0]],
    [[ 4,  4,  1], [ 3, -5,  0], [ 5,  2,  2], [ 4,  3, -1], [ 3, -4,  1]],
    [[ 5, -2,  1], [ 4,  4, -1], [ 3, -3,  2], [ 6,  1,  0], [ 4,  2,  1]],
    [[ 3,  4,  2], [ 5, -1, -1], [ 4,  3,  0], [ 3, -5,  1], [ 5,  2,  2]],
    [[ 4, -3,  1], [ 3,  5,  0], [ 6, -1,  2], [ 4,  2, -1], [ 3, -4,  1]],
    [[ 5,  2, -1], [ 4, -3,  1], [ 3,  5,  2], [ 5,  1,  0], [ 4, -2, -1]],
    [[ 3, -5,  0], [ 5,  2,  1], [ 4,  3, -1], [ 6, -1,  2], [ 3,  4,  0]],
    [[ 4,  3,  2], [ 3, -4,  0], [ 5,  1, -1], [ 4,  4,  1], [ 3, -3,  2]],
    [[ 5, -1,  0], [ 4,  3,  2], [ 3, -5,  1], [ 5,  2, -1], [ 4, -2,  0]],
    [[ 3,  4, -1], [ 5, -2,  1], [ 4,  3,  0], [ 3, -4,  2], [ 5,  1, -1]],
    [[ 4, -3,  2], [ 3,  5,  0], [ 5, -1,  1], [ 4,  2, -1], [ 3, -5,  2]],
    [[ 5,  2,  1], [ 4, -4, -1], [ 3,  3,  2], [ 5,  1,  0], [ 4, -2,  1]],
    [[ 3, -4,  0], [ 5,  2,  1], [ 4,  3, -1], [ 3, -5,  2], [ 5,  1,  0]],
    [[ 4,  3,  1], [ 3, -4,  2], [ 5,  2, -1], [ 4,  4,  0], [ 3, -3,  1]],
    [[ 5, -2,  0], [ 4,  3,  1], [ 3, -5,  2], [ 5,  1, -1], [ 4,  2,  0]],
    [[ 3,  5,  1], [ 5, -1, -1], [ 4,  2,  2], [ 3, -4,  0], [ 5,  3,  1]],
    [[ 4, -3,  2], [ 3,  4, -1], [ 5,  1,  0], [ 4, -5,  1], [ 3,  2,  2]],
    [[ 5,  2, -1], [ 4, -4,  1], [ 3,  5,  0], [ 5,  1,  2], [ 4, -2, -1]],
    [[ 3, -5,  1], [ 5,  2,  0], [ 4,  3, -1], [ 3, -4,  2], [ 5,  1,  1]],
    [[ 4,  3,  0], [ 3, -5,  2], [ 5,  1, -1], [ 4,  2,  1], [ 3, -4,  0]],
    [[ 5, -2,  1], [ 4,  4, -1], [ 3, -3,  2], [ 5,  1,  0], [ 4,  2,  1]],
    [[ 3,  5, -1], [ 5, -1,  1], [ 4,  2,  0], [ 3, -4,  2], [ 5,  3, -1]],
    [[ 4, -3,  2], [ 3,  4,  0], [ 5,  1, -1], [ 4, -5,  1], [ 3,  2,  2]],
    [[ 5,  2,  1], [ 4, -4, -1], [ 3,  5,  0], [ 5,  1,  2], [ 4, -2,  1]],
    [[ 3, -5,  0], [ 5,  2,  1], [ 4,  3, -1], [ 3, -4,  2], [ 5,  1,  0]],
    [[ 4,  3,  2], [ 3, -4,  0], [ 5,  1, -1], [ 4,  4,  1], [ 3, -3,  2]],
    [[ 5, -2,  1], [ 4,  3,  0], [ 3, -5,  2], [ 5,  1, -1], [ 4,  2,  1]]
], dtype=tf.float32)

# -----------------------------
# DEFINE CAMERAS
# -----------------------------
cameras = {
    "cam1": Camera(position=[120, 5, 50], look_at=[15, 0, 0]),
    "cam2": Camera(position=[0, 0, 220], look_at=[0, 0, 0])
}

displacements = tf.cumsum(velocities * dt, axis=1, exclusive=False)
transmitter_path = tf.concat([start_points[:, tf.newaxis, :],
                               start_points[:, tf.newaxis, :] + displacements], axis=1)
scene.preview()"""

In [ ]:


# -----------------------------
# RUN SIMULATION
# -----------------------------
kpi_log = simulate(
    rx_path=rx_path,
    cameras=cameras,
    scene=scene,
    generate_metrics=generate_metrics,
    render=render
)

In [ ]:
print(scene.transmitters.keys())
print(scene.receivers.keys())

In [ ]:
scene.preview()